# blueshark · Tier-0 training (Colab GPU)

Clone repo → pull agentic corpus → train a tokenizer → pack → **masked SFT** → sample.
Checkpoints save to Google Drive so the model can be pulled back to the laptop.

Pipeline: `pipeline/parse_swe_trajectories.py` → `tokenize_pack.py` → `train.py` (streamed, masked SFT, warmup+cosine, eval, resume, logging).


## 1 · GPU + setup

In [ ]:
!nvidia-smi -L || echo 'no GPU - set Runtime > Change runtime type > GPU'

In [ ]:
import os
REPO='/content/blueshark'
if not os.path.exists(REPO):
    !git clone -b feat/training-pipeline https://github.com/hackspaces/blueshark.git {REPO}
else:
    !cd {REPO} && git pull --ff-only
%cd {REPO}
!pip -q install tokenizers pyarrow
print('torch:', __import__('torch').__version__, '| cuda:', __import__('torch').cuda.is_available())


## 2 · Mount Drive (checkpoints land here so we can pull them home)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR='/content/drive/MyDrive/blueshark/runs/sft_proof'
os.makedirs(RUN_DIR, exist_ok=True)
print('checkpoints ->', RUN_DIR)


## 3 · Pull the agentic corpus (Nebius SWE-agent-trajectories)
Start with a few shards for speed; raise `N_SHARDS` (max 12) for the full 80k set.

In [ ]:
import urllib.request
N_SHARDS = 4   # each ~90MB; 12 = full 80k trajectories
os.makedirs('data/raw/swe_trajectories', exist_ok=True)
base='https://huggingface.co/datasets/nebius/SWE-agent-trajectories/resolve/main/data/'
for i in range(N_SHARDS):
    fn=f'train-{i:05d}-of-00012.parquet'; dst='data/raw/swe_trajectories/'+fn
    if not os.path.exists(dst):
        print('downloading', fn); urllib.request.urlretrieve(base+fn, dst)
print('shards:', len(os.listdir('data/raw/swe_trajectories')))


## 4 · Parse → token format with loss masks

In [ ]:
!python pipeline/parse_swe_trajectories.py

## 5 · Train a byte-level BPE tokenizer on the corpus (keeps the 5 agentic tokens)

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
SPECIAL=['<plan>','<think>','</think>','<tool_call>','<tool_result>']
if not os.path.exists('data/tokenizer.json'):
    tok=Tokenizer(models.BPE()); tok.pre_tokenizer=pre_tokenizers.ByteLevel(add_prefix_space=False); tok.decoder=decoders.ByteLevel()
    tr=trainers.BpeTrainer(vocab_size=8192, special_tokens=SPECIAL,
                           initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=True)
    tok.train(['data/sft/swe_agentic.txt'], tr); tok.save('data/tokenizer.json')
print('vocab:', Tokenizer.from_file('data/tokenizer.json').get_vocab_size())


## 6 · Pack to streamable shards

In [ ]:
!python pipeline/tokenize_pack.py --sft data/sft/swe_agentic_sft.jsonl --name swe_sft

## 7 · Train (masked SFT, checkpoints to Drive)
Resumes automatically if the session drops — re-run this cell with `--resume`.

In [ ]:
!python pipeline/train.py --data swe_sft --config proof --out "{RUN_DIR}" \
   --steps 3000 --batch 24 --seq 512 --lr 3e-3 \
   --log-every 25 --eval-every 200 --save-every 200


## 8 · Loss curve

In [ ]:
import json, matplotlib.pyplot as plt
rows=[json.loads(l) for l in open(f'{RUN_DIR}/metrics.jsonl')]
tr=[(r['step'],r['train_loss']) for r in rows if 'train_loss' in r]
va=[(r['step'],r['val_loss']) for r in rows if 'val_loss' in r]
plt.figure(figsize=(8,4)); plt.plot(*zip(*tr),alpha=.6,label='train')
if va: plt.plot(*zip(*va),'o-',label='val')
plt.xlabel('step'); plt.ylabel('masked loss'); plt.legend(); plt.grid(alpha=.3); plt.show()


## 9 · Sample from the trained model

In [ ]:
import torch, sys; sys.path.insert(0,'.')
from model import Model; from configs import build_config
from tokenizers import Tokenizer
tok=Tokenizer.from_file('data/tokenizer.json')
ck=torch.load(f'{RUN_DIR}/best.pt', map_location='cuda')
cfg=build_config(ck['config'], tok.get_vocab_size())
m=Model(cfg).cuda(); m.load_state_dict(ck['model']); m.eval()
prompt="We're currently solving the following issue within our repository.\nISSUE:\nFix the off-by-one error in the paginator.\n<think>"
x=torch.tensor([tok.encode(prompt).ids]).cuda()
for _ in range(150):
    with torch.no_grad(): logits,_=m(x[:,-cfg.max_seq:])
    nxt=torch.multinomial(torch.softmax(logits[0,-1]/0.8,-1),1)
    x=torch.cat([x,nxt.view(1,1)],1)
print(tok.decode(x[0].tolist()))


## Notes
- **Resume after a disconnect:** re-run cells 1-2, then run cell 7 with `--resume` appended.
- **Bring it home:** `best.pt` + `tokenizer.json` are on Drive under `MyDrive/blueshark/runs/sft_proof`; pull them to `~/blueshark` to test in the local viewer.
- **Scale up:** raise `N_SHARDS` to 12, `--steps`, `--config finegrained`. Watch val loss in cell 8 to catch overfitting.
